# 02 – FDG PET Segmentation Practice — Solution

**SNMMI AI Certificate – Hands-On Module**

This is the **complete solution** to the FDG practice notebook.
All three tasks are implemented with working code.

> **Run cells top-to-bottom on a fresh Colab runtime.**

---

**Dataset citation:**

> Gatidis, S., Kuestner, T., Ingrisch, M., Hepp, T., Frueh, M., Nikolaou, K.,
> La Fougere, C., Pfannenberg, C., Fabritius, M., Jeblick, K., Schachtner, B.,
> Dexl, J., Wesp, P., Mittermeier, A., Unterrainer, L., Sheikh, G., Boening, G.,
> Brendel, M., Ricke, J., … Cyran, C. (2025).
> *PSMA-FDG-PET-CT-Lesions*.
> University of Tuebingen, Ludwig-Maximilians-University Munich.
> [https://doi.org/10.57754/FDAT.0zs4c-89f12](https://doi.org/10.57754/FDAT.0zs4c-89f12)
>
> License: **CC BY-NC 4.0** – non-commercial use only.

## 0 · Setup & Config

In [ ]:
%pip install -q numpy matplotlib tqdm scikit-learn

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

EPOCHS     = 128
BATCH_SIZE = 16
LR         = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1 · Download & Unzip FDG Sample Data

In [ ]:
import pathlib, urllib.request, zipfile

REPO   = "ZekunLi-Zeke/snmmi-ai-hands-on-segmentation-pub"
TRACER = "fdg"

DATA_DIR = pathlib.Path("../data") / TRACER
DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_path = pathlib.Path(f"/tmp/{TRACER}.zip")

# Use the /releases/latest/download/ redirect so new data releases are picked
# up automatically without editing this cell.
DATA_URL = (
    f"https://github.com/{REPO}/releases/latest/download/{TRACER}.zip"
)

if not any(DATA_DIR.rglob("*.npz")):
    print(f"Downloading {TRACER}.zip …")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    print("Unzipping …")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already present.")

# Locate the pre-split npz files (works regardless of nesting inside the zip)
train_npz = next(DATA_DIR.rglob("train_patches.npz"), None)
val_npz   = next(DATA_DIR.rglob("val_patches.npz"),   None)
assert train_npz and val_npz, (
    "train_patches.npz / val_patches.npz not found – check the zip contents."
)
print(f"Train file : {train_npz}")
print(f"Val   file : {val_npz}")

# Quick peek at array shapes
for label, path in [("train", train_npz), ("val", val_npz)]:
    d = np.load(path)
    print(f"  {label}: X{d['X'].shape}  Y{d['Y'].shape}  "
          f"(dtype X={d['X'].dtype})")


## 2 · Load Data & Visualise (Solution: augmentation ON)

In [ ]:
# Task 2 solution: AUGMENT = True (horizontal flip enabled)
AUGMENT = True

class PatchDataset(Dataset):
    """2-D PET patch dataset loaded from a .npz file.

    Expected .npz keys
    ------------------
    X : (N, 1, 32, 32) float16 – normalised PET image.
    Y : (N, 1, 32, 32) uint8   – binary lesion mask {0, 1}.
    """

    def __init__(self, npz_path, augment=False):
        d = np.load(npz_path)
        self.X = d["X"].astype(np.float32)   # (N, C, H, W)
        self.Y = d["Y"].astype(np.float32)   # (N, 1, H, W)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].copy()   # (C, H, W)
        y = self.Y[idx].copy()   # (1, H, W)

        if self.augment and random.random() > 0.5:
            # Task 2: horizontal flip augmentation
            x = x[:, :, ::-1].copy()   # flip all channels
            y = y[:, :, ::-1].copy()   # flip mask

        return torch.from_numpy(x), torch.from_numpy(y)


train_ds = PatchDataset(train_npz, augment=AUGMENT)
val_ds   = PatchDataset(val_npz,   augment=False)
n_train  = len(train_ds)
n_val    = len(val_ds)
n_ch     = train_ds.X.shape[1]   # 1 = PET only
print(f"Dataset: train {n_train}  val {n_val}  channels {n_ch}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Visualise a few patches
n_show_cols = min(4, n_train)
fig, axes = plt.subplots(n_ch + 1, n_show_cols, figsize=(3 * n_show_cols, 3 * (n_ch + 1)))
axes = np.atleast_2d(axes)
ch_labels = ["PET"]
for col in range(n_show_cols):
    x, y = train_ds[col]
    for c, lbl in enumerate(ch_labels):
        cmap = "hot" if lbl == "PET" else "gray"
        axes[c, col].imshow(x[c].numpy(), cmap=cmap)
        axes[c, col].set_title(f"{lbl} #{col}")
    axes[-1, col].imshow(y[0].numpy(), cmap="gray")
    axes[-1, col].set_title(f"Mask #{col}")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Sample patches (PET / Mask)", fontsize=13)
plt.tight_layout()
plt.show()


## 3 · Define Compact 2-D U-Net

### Architecture Overview

The **U-Net** (Ronneberger et al., 2015) is an **encoder–decoder** CNN with **skip connections** at each resolution level. It was designed for biomedical image segmentation and excels even on small datasets by reusing low-level spatial features during decoding.

**Three structural parts:**

| Part | Mechanism | Purpose |
|------|-----------|---------|
| **Encoder** | `DoubleConv` → `MaxPool2d` | Halves spatial size, doubles channels — builds increasingly abstract feature maps |
| **Bottleneck** | `DoubleConv` | Deepest, most compressed representation at the coarsest spatial scale |
| **Decoder** | `ConvTranspose2d` + `DoubleConv` | Doubles spatial size, halves channels — reconstructs spatial detail for the segmentation mask |

**Skip connections** copy each encoder feature map and concatenate it *channel-wise* with the matching decoder feature map. This allows the network to combine high-level semantic context (from the bottleneck) with fine spatial detail (from the encoder) — critical for accurate segmentation boundaries.

### Building Block: `DoubleConv`

Each encoder/decoder level applies two consecutive Conv→BN→ReLU operations:

$$\mathbf{x} \;\xrightarrow{\,\text{Conv}_{3\times3}\,}\; \xrightarrow{\,\text{BN}\,}\; \xrightarrow{\,\text{ReLU}\,}\; \mathbf{h} \;\xrightarrow{\,\text{Conv}_{3\times3}\,}\; \xrightarrow{\,\text{BN}\,}\; \xrightarrow{\,\text{ReLU}\,}\; \mathbf{out}$$

- **Conv 3×3 (padding=1):** extracts local spatial features; padding=1 keeps H×W unchanged.
- **BatchNorm:** normalises activations per channel across the batch → faster convergence, less sensitivity to weight initialisation.
- **ReLU:** zeroes negative values, introducing the non-linearity needed to learn complex mappings.

### Network Shape at a Glance (features=16, 32×32 input)

```
 Input  (B,  1, 32, 32)
   | enc1 ─────────────────────────────────────────── skip ──|
   | (B, 16, 32, 32) <- e1                                   |
   | MaxPool2d v                                             |
   | enc2 ─────────────────────────── skip ──|              |
   | (B, 32, 16, 16) <- e2                   |              |
   | MaxPool2d v                              |              |
   | enc3 ─────────── skip ──|               |              |
   | (B, 64,  8,  8) <- e3   |               |              |
   | MaxPool2d v              |               |              |
   | bottleneck               |               |              |
   | (B,128,  4,  4) <- b     |               |              |
   | up3 ^ (x2)               |               |              |
   | cat([up3(b), e3]) dec3 <-|               |              |
   | (B, 64,  8,  8) <- d3                    |              |
   | up2 ^ (x2)                               |              |
   | cat([up2(d3), e2]) dec2 <----------------|              |
   | (B, 32, 16, 16) <- d2                                   |
   | up1 ^ (x2)                                              |
   | cat([up1(d2), e1]) dec1 <-------------------------------|
   | (B, 16, 32, 32) <- d1
   | head  (1x1 conv)
   +-> logits  (B,  1, 32, 32)   <- raw values, sigmoid applied later
```

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3x3 conv -> BN -> ReLU blocks.

    This is the fundamental building block used at every level of the U-Net.
    Two stacked 3x3 convolutions increase the effective receptive field while
    keeping spatial dimensions (H x W) unchanged via padding=1.
    """

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            # 1st Conv: maps in_ch feature channels -> out_ch.
            # kernel=3, padding=1 -> output H x W equals input H x W.
            # bias=False: the following BatchNorm already provides a learnable bias term.
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            # BatchNorm: normalises each channel's activations across the batch.
            # Reduces internal covariate shift -> faster convergence, better gradient flow.
            nn.BatchNorm2d(out_ch),
            # ReLU: zeroes all negative activations, introducing non-linearity.
            # inplace=True modifies the tensor in place to reduce peak memory usage.
            nn.ReLU(inplace=True),
            # 2nd Conv: out_ch -> out_ch (same channel count).
            # Two stacked 3x3 convs give a 5x5 effective receptive field while
            # using fewer parameters than a single 5x5 convolution.
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)   # pass input sequentially through all layers


class UNet2D(nn.Module):
    """Compact 2-D U-Net with skip connections (4 resolution levels)."""

    def __init__(self, in_ch=1, out_ch=1, features=16):
        super().__init__()
        f = features   # base channel count; doubles at each encoder level

        # -- Encoder -----------------------------------------------------------
        # Three DoubleConv blocks extract features at decreasing spatial resolutions.
        # Between blocks a shared MaxPool2d (below) halves H and W.
        self.enc1 = DoubleConv(in_ch, f)       # (B, in_ch, 32, 32) -> (B,  f, 32, 32)
        self.enc2 = DoubleConv(f,     f * 2)   # (B,  f,    16, 16) -> (B, 2f, 16, 16)
        self.enc3 = DoubleConv(f * 2, f * 4)   # (B, 2f,     8,  8) -> (B, 4f,  8,  8)
        # MaxPool2d(2): halves spatial size -- no learned parameters.
        self.pool  = nn.MaxPool2d(2)

        # -- Bottleneck --------------------------------------------------------
        # Deepest layer: most abstract, spatially coarsest representation.
        self.bottleneck = DoubleConv(f * 4, f * 8)   # (B, 4f, 4, 4) -> (B, 8f, 4, 4)

        # -- Decoder -----------------------------------------------------------
        # Each level: ConvTranspose2d doubles H x W (learned upsampling),
        # followed by concatenation with the encoder skip connection and a DoubleConv.
        # The channel width doubles after concatenation, hence the wider dec* inputs.
        self.up3  = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(f * 8, f * 4)   # 4f (up) + 4f (skip) = 8f in
        self.up2  = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(f * 4, f * 2)   # 2f (up) + 2f (skip) = 4f in
        self.up1  = nn.ConvTranspose2d(f * 2, f,     kernel_size=2, stride=2)
        self.dec1 = DoubleConv(f * 2, f)        #  f (up) +  f (skip) = 2f in

        # Output head: 1x1 conv -> out_ch raw logits (sigmoid applied in loss/metric).
        self.head = nn.Conv2d(f, out_ch, 1)

    def forward(self, x):
        # -- Encoder pass ------------------------------------------------------
        e1 = self.enc1(x)              # (B,  f, 32, 32)
        e2 = self.enc2(self.pool(e1))  # pool -> halve; enc2 -> (B, 2f, 16, 16)
        e3 = self.enc3(self.pool(e2))  # pool -> halve; enc3 -> (B, 4f,  8,  8)

        # -- Bottleneck --------------------------------------------------------
        b  = self.bottleneck(self.pool(e3))  # (B, 8f, 4, 4)

        # -- Decoder pass (upsample + concatenate skip + convolve) -------------
        # torch.cat(..., dim=1): concatenate along the channel axis.
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))  # (B, 4f,  8,  8)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # (B, 2f, 16, 16)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # (B,  f, 32, 32)

        return self.head(d1)   # (B, 1, H, W) raw logits


# n_ch was detected from the data in the previous cell
model    = UNet2D(in_ch=n_ch, out_ch=1, features=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net  in_ch={n_ch}  parameters: {n_params:,}")


## 4 · Define Loss & Optimizer (Solution: Dice + BCE)

### Loss Functions

**Dice Loss** measures segmentation overlap using soft probabilities $p = \sigma(\text{logit})$:

$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2\displaystyle\sum_i p_i \, g_i + \varepsilon}{\displaystyle\sum_i p_i + \sum_i g_i + \varepsilon}$$

$\varepsilon$ (`smooth`) prevents division by zero for empty masks. $\mathcal{L}_{\text{Dice}} = 0$ for perfect overlap.

**BCE** provides stable per-pixel gradients even for sparse tumour masks:

$$\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_i \left[ g_i \log p_i + (1 - g_i)\log(1 - p_i) \right]$$

**Combined:** $\mathcal{L} = \mathcal{L}_{\text{Dice}} + \mathcal{L}_{\text{BCE}}$

In [ ]:
# Task 1 solution: Dice + BCE combined loss
def dice_loss(logits, target, smooth=1.0):
    # Step 1 -- convert raw logits to probabilities in [0, 1] via sigmoid.
    pred  = torch.sigmoid(logits)
    # Step 2 -- soft intersection: element-wise product summed over H, W.
    inter = (pred * target).sum(dim=(2, 3))
    # Step 3 -- denominator: sum of predicted area + GT area.
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    # Step 4 -- Dice coefficient per sample -> mean over batch -> convert to loss.
    return 1.0 - ((2.0 * inter + smooth) / (denom + smooth)).mean()


def loss_fn(logits, target):
    """Task 1 solution: Dice loss + Binary Cross-Entropy."""
    # BCE: numerically stable per-pixel cross-entropy computed directly on logits.
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    # Combine: Dice for overlap quality, BCE for stable gradient signal.
    return dice_loss(logits, target) + bce


optimizer = optim.Adam(model.parameters(), lr=LR)
print("Loss: Dice + BCE  |  Optimizer: Adam  |  LR:", LR)

## 5 · Training Loop

### Training Loop Structure

Each epoch runs two phases:

1. **Training phase** (`model.train()`): forward pass → loss → `loss.backward()` → `optimizer.step()` → `optimizer.zero_grad()`.
2. **Validation phase** (`model.eval()` + `torch.no_grad()`): Dice on the held-out set without gradient tracking.

In [ ]:
def dice_score(logits, target, threshold=0.5):
    """Dice coefficient metric (non-differentiable, used for reporting).

    Unlike dice_loss -- which uses soft sigmoid probabilities for backpropagation --
    this function binarises predictions at a hard threshold before computing Dice.
    This gives an interpretable, clinically meaningful overlap score.
    """
    # Apply sigmoid to convert raw logits -> probabilities, then threshold to 0/1.
    # A pixel is predicted as tumour if its probability exceeds 0.5.
    pred  = (torch.sigmoid(logits) > threshold).float()
    # Count true positives (overlap): pixels labelled tumour in both pred and target.
    # Sum over spatial dims (2 = H, 3 = W) independently per sample in the batch.
    inter = (pred * target).sum(dim=(2, 3))
    # Denominator = total predicted area + total GT area (Dice "union" term).
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    # Dice = 2|A intersect B| / (|A| + |B|); smooth=1.0 avoids 0/0 for empty GT masks.
    return ((2.0 * inter + 1.0) / (denom + 1.0)).mean().item()


history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    model.train()   # enable batch-norm training behaviour
    epoch_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()          # reset accumulated gradients
        loss = loss_fn(model(x), y)
        loss.backward()                # backprop: compute gradients for all params
        optimizer.step()               # Adam weight update
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= n_train

    model.eval()
    val_dice_sum = 0.0
    with torch.no_grad():              # no gradient tracking in validation
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            val_dice_sum += dice_score(model(x), y) * x.size(0)
    val_dice = val_dice_sum / n_val

    history["train_loss"].append(epoch_loss)
    history["val_dice"].append(val_dice)
    print(f"Epoch {epoch:>2}: train loss={epoch_loss:.4f}  val Dice={val_dice:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(range(1, EPOCHS + 1), history["train_loss"], marker="o")
ax1.set(title="Train Loss", xlabel="Epoch", ylabel="Loss")
ax2.plot(range(1, EPOCHS + 1), history["val_dice"], marker="o", color="orange")
ax2.set(title="Validation Dice", xlabel="Epoch", ylabel="Dice")
plt.tight_layout()
plt.show()

## 6 · Visualise Predictions

In [ ]:
model.eval()
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 3, figsize=(10, 3.5 * n_show))

with torch.no_grad():
    for row in range(n_show):
        x, y  = val_ds[row]
        logit = model(x.unsqueeze(0).to(DEVICE))
        pred  = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()
        pet_ch = x[-1].numpy()   # PET channel
        axes[row, 0].imshow(pet_ch, cmap="hot");  axes[row, 0].set_title("PET")
        axes[row, 1].imshow(y[0].numpy(), cmap="gray"); axes[row, 1].set_title("GT Mask")
        axes[row, 2].imshow(pred,          cmap="gray"); axes[row, 2].set_title("DL Prediction")

for ax in axes.flat: ax.axis("off")
plt.suptitle("Predictions vs Ground Truth", fontsize=14)
plt.tight_layout(); plt.show()


## 7 · Task 3 Solution – Dice & Tumor-Area Bias

In [ ]:
def threshold_segment(pet_patch, fraction=0.4):
    """Threshold segmentation at *fraction* × per-patch max SUV."""
    max_suv = pet_patch.max()
    if max_suv == 0:
        return np.zeros_like(pet_patch)
    return (pet_patch > fraction * max_suv).astype(np.float32)


def arr_dice(a, b, smooth=1.0):
    inter = (a * b).sum()
    return (2.0 * inter + smooth) / (a.sum() + b.sum() + smooth)


# ── Task 3 solution ───────────────────────────────────────────────────────────
model.eval()
all_dice, thr_dices = [], []
dl_biases, thr_biases = [], []

with torch.no_grad():
    for i in range(len(val_ds)):
        x, y    = val_ds[i]
        pet_np  = x[-1].numpy()   # last channel is PET
        gt_mask = y[0].numpy()

        logit   = model(x.unsqueeze(0).to(DEVICE))
        dl_pred = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()
        thr_mask = threshold_segment(pet_np)

        # Task 3a: Dice
        all_dice.append(arr_dice(dl_pred, gt_mask))
        thr_dices.append(arr_dice(thr_mask, gt_mask))

        # Task 3b: tumor-area bias (%)
        gt_area = gt_mask.sum()
        if gt_area > 0:
            dl_biases.append((dl_pred.sum() - gt_area) / gt_area * 100.0)
            thr_biases.append((thr_mask.sum() - gt_area) / gt_area * 100.0)

# Task 3c: print summary
print(f"Mean Dice (DL):             {np.mean(all_dice):.3f}")
print(f"Mean Dice (40% thr):        {np.mean(thr_dices):.3f}")
print(f"Mean DL Tumor-Area Bias:    {np.mean(dl_biases):+.1f} %")
print(f"Mean Thr Tumor-Area Bias:   {np.mean(thr_biases):+.1f} %")

# Visualise DL vs threshold for one sample (sample 9 shows clear contrast)
i = 9
x, y     = val_ds[i]
pet_np   = x[-1].numpy()
gt_mask  = y[0].numpy()
thr_mask = threshold_segment(pet_np)
with torch.no_grad():
    dl_pred = (torch.sigmoid(model(x.unsqueeze(0).to(DEVICE))) > 0.5).float().cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(pet_np,  cmap="hot");  axes[0].set_title("PET input")
axes[1].imshow(gt_mask, cmap="gray"); axes[1].set_title("GT Mask")
axes[2].imshow(dl_pred, cmap="gray"); axes[2].set_title(f"DL  (Dice={all_dice[i]:.2f})")
axes[3].imshow(thr_mask,cmap="gray"); axes[3].set_title(f"40% Thr (Dice={thr_dices[i]:.2f})")
for ax in axes: ax.axis("off")
plt.suptitle("DL Model vs 40 % SUV Threshold Baseline", fontsize=13)
plt.tight_layout(); plt.show()